# Stage 5 · Consolidated Results Export

Rebuilds all analysis frames from cached state and exports every number,
series, and coefficient reported in the paper into a single archive
(`paper_data.zip`), including recomputed event-study coefficients and
robustness grids. See MANIFEST.md for the statistic-to-file mapping.

---
## Cell 1 — Setup and cache inventory

Mounts Drive and verifies every input exists before anything is computed, so a missing file is reported here rather than as a stack trace later.

In [ ]:
# ===================== CELL 1 — SETUP + INVENTORY =======================
!pip -q install linearmodels rdrobust pandas numpy scipy pyarrow 2>/dev/null

import os, json, zipfile, subprocess, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd

from google.colab import drive; drive.mount('/content/drive')
WORK  = Path('/content/drive/MyDrive/msr2027')
CACHE = WORK/'cache'
EXP   = CACHE/'paper_export'; EXP.mkdir(parents=True, exist_ok=True)

needed = {
 'ranking'      : sorted(CACHE.glob('ranking_*.parquet')),
 'rdd_frame'    : [CACHE/'analysis_frame_rdd.parquet'],
 'late_entrants': [CACHE/'late_entrants.parquet'],
 'panel_state'  : [CACHE/'state'/'pypi_month_panel.jsonl'],
 'outcome_state': [CACHE/'state'/'pypi_outcomes.jsonl'],
 'results_d2'   : [CACHE/'results_rdd.json'],
 'results_d3'   : [CACHE/'results_panel.json'],
 'results_d4'   : [CACHE/'results_extended.json'],
}
optional = {
 'prespec'      : sorted(CACHE.glob('PRESPEC_*.json')),
 'fig_trends'   : [CACHE/'fig_raw_trends.png'],
 'fig_event'    : [CACHE/'fig_event_study.png'],
}
missing=[]
for k,v in needed.items():
    ok = v and v[-1].exists()
    print(f'  [{"OK " if ok else "MISS"}] {k:14s} {v[-1] if v else "(none)"}')
    if not ok: missing.append(k)
for k,v in optional.items():
    ok = v and v[-1].exists()
    print(f'  [{"OK " if ok else "--- "}] {k:14s} (optional)')
if missing:
    raise SystemExit(f'>>> Missing required inputs: {missing}. '
                     'Re-run the notebook that produces them, then return here.')
INV_DONE=True; print('\nCELL 1 OK — all required inputs present')

---
## Cell 2 — Rebuild and recompute

Rebuilds the RDD frame and monthly panel from cache, then recomputes the never-saved pieces: event-study coefficient vector, robustness grids, and stage-1 validation statistics.

In [ ]:
# ================== CELL 2 — REBUILD + RECOMPUTE ========================
if not globals().get('INV_DONE'): raise SystemExit('>>> Run CELL 1 first.')
from linearmodels.panel import PanelOLS
from rdrobust import rdrobust
from scipy.stats import spearmanr

def read_jsonl(p):
    rows=[]
    with open(p) as f:
        for line in f:
            try: rows.append(json.loads(line))
            except Exception: pass
    df=pd.DataFrame(rows)
    return df.rename(columns={'_key':'project'}).drop_duplicates('project', keep='last')

rankdf = pd.read_parquet(sorted(CACHE.glob('ranking_*.parquet'))[-1])
full   = pd.read_parquet(CACHE/'analysis_frame_rdd.parquet')
late   = set(pd.read_parquet(CACHE/'late_entrants.parquet')['project'])
gotm   = read_jsonl(CACHE/'state'/'pypi_month_panel.jsonl')
raw2   = read_jsonl(CACHE/'state'/'pypi_outcomes.jsonl')
CUT    = float(rankdf.loc[rankdf['rank']==3800,'num_downloads'].iloc[0])
full['released_at_all'] = (full['n_rel_post']>0).astype(float)
frame  = full[~full['project'].isin(late)].copy()

# ---- monthly panel -----------------------------------------------------
months = pd.period_range('2021-01','2023-12',freq='M').astype(str).tolist()
t0i = months.index('2022-07')
mset = {p:set(m) for p,m in zip(gotm['project'], gotm['months'])}
tr   = dict(zip(frame['project'], frame['treated']))
rows=[]
for p in frame['project']:
    if p not in mset: continue
    s=mset[p]; t=tr[p]
    for i,mm in enumerate(months): rows.append((p,i,mm,int(mm in s),int(t)))
panel = pd.DataFrame(rows, columns=['project','mnum','month','released','treated'])
print(f'panel rebuilt: {len(panel):,} rows, {panel.project.nunique():,} projects')

# ---- event-study coefficients (never saved before) ---------------------
pan = panel.copy(); pan['k']=pan['mnum']-t0i
ev  = pan[(pan.k>=-12)&(pan.k<=12)].set_index(['project','mnum'])
X   = pd.DataFrame({f'k{k}': ((ev['k']==k)&(ev['treated']==1)).astype(float)
                    for k in range(-12,13) if k!=-1}, index=ev.index)
mE  = PanelOLS(ev['released'], X, entity_effects=True, time_effects=True
      ).fit(cov_type='clustered', cluster_entity=True)
ci  = mE.conf_int()
es  = pd.DataFrame({'k':[k for k in range(-12,13) if k!=-1]})
es['coef']=[float(mE.params[f'k{k}']) for k in es.k]
es['lo']  =[float(ci.loc[f'k{k}','lower']) for k in es.k]
es['hi']  =[float(ci.loc[f'k{k}','upper']) for k in es.k]
es['p']   =[float(mE.pvalues[f'k{k}']) for k in es.k]
print('event-study coefficients recomputed')

# ---- robustness grids --------------------------------------------------
def _rd(y, x, **kw):
    # Guarded: a placebo cutoff whose download value falls outside the band
    # leaves all observations on one side of zero; record NaN, don't crash.
    try:
        y=np.asarray(y,float); x=np.asarray(x,float)
        ok=np.isfinite(y)&np.isfinite(x); y,x=y[ok],x[ok]
        if len(y)<50 or x.min()>=0 or x.max()<=0:
            return float('nan'), float('nan'), 0
        r=rdrobust(y=y, x=x, c=0, **kw)
        return float(r.coef.iloc[0,0]), float(r.pv.iloc[2,0]), int(np.sum(np.asarray(r.N_h)))
    except Exception:
        return float('nan'), float('nan'), 0
grid=[]
for pr in [1500,2000,2500,3000,5000,6000,7000,8000]:
    s=rankdf.loc[rankdf['rank']==pr,'num_downloads']
    if not len(s): continue
    dd=full.assign(r2=np.log(full.num_downloads)-np.log(float(s.iloc[0])))
    dd=dd[(dd.r2>-0.7)&(dd.r2<0.7)]
    e,p_,n = _rd(dd['released_at_all'],dd['r2']) if len(dd)>=150 else (float('nan'),float('nan'),0)
    grid.append(('placebo_cutoff',pr,e,p_,n))
for d in [0.0,0.02,0.05,0.10]:
    dd=full[np.abs(full['running'])>=d]; e,p_,n=_rd(dd['released_at_all'],dd['running'])
    grid.append(('donut',d,e,p_,n))
for h in [0.15,0.25,0.35,0.50,0.70]:
    dd=full[np.abs(full['running'])<=h]; e,p_,n=_rd(dd['released_at_all'],dd['running'],h=h)
    grid.append(('bandwidth',h,e,p_,n))
for cs in [3500,3600,3700,3800,3900,4000]:
    s=rankdf.loc[rankdf['rank']==cs,'num_downloads']
    dd=full.assign(r2=np.log(full.num_downloads)-np.log(float(s.iloc[0])))
    dd=dd[(dd.r2>-0.7)&(dd.r2<0.7)]; e,p_,n=_rd(dd['released_at_all'],dd['r2'])
    grid.append(('cohort_size',cs,e,p_,n))
grids=pd.DataFrame(grid,columns=['test','param','est','p','n_eff'])
print('robustness grids:', len(grids), 'rows,',
      int(grids.est.isna().sum()), 'NaN (placebo cutoffs outside band — expected)')

# ---- stage-1 validation + density + always-takers ------------------------
if not Path('/content/tpp').exists():
    subprocess.run(['git','clone','-q','--filter=blob:none',
                    'https://github.com/hugovk/top-pypi-packages.git','/content/tpp'])
snap=subprocess.run(['git','-C','/content/tpp','show',
                     'dd326bf:top-pypi-packages-30-days.min.json'],
                    capture_output=True,text=True).stdout
tpp=pd.DataFrame(json.loads(snap)['rows']); tpp['tpp_rank']=np.arange(1,len(tpp)+1)
mm=rankdf.merge(tpp[['project','tpp_rank']],on='project',how='inner')
near=mm[(mm['rank']>1900)&(mm['rank']<5700)]
H=0.15
def _ratio(c):
    run=np.log(rankdf['num_downloads'])-np.log(c)
    lo=int(((run>-H)&(run<0)).sum()); hi=int(((run>=0)&(run<H)).sum())
    return lo,hi,(hi/lo if lo else float('nan'))
LO,HI,RATIO=_ratio(CUT)
plac=[]
for pr in range(1140,9500,50):
    if abs(pr-3800)<500: continue
    s=rankdf.loc[rankdf['rank']==pr,'num_downloads']
    if not len(s): continue
    l,h2,r2=_ratio(float(s.iloc[0]))
    if np.isfinite(r2) and l>20: plac.append(r2)
med=float(np.median(plac))
validation={'rho_all':float(spearmanr(mm['rank'],mm['tpp_rank']).correlation),
 'rho_near':float(spearmanr(near['rank'],near['tpp_rank']).correlation),
 'overlap_n':int(len(mm)),
 'treat_agree':float(((mm['rank']<=3800)==(mm['tpp_rank']<=3800)).mean()),
 'density_true_ratio':float(RATIO),'density_lo':LO,'density_hi':HI,
 'density_placebo_median':med,'density_n_placebo':len(plac),
 'density_p':float((np.abs(np.array(plac)-med)>=abs(RATIO-med)).mean()),
 'cutoff_downloads':CUT}
print('validation:', json.dumps(validation, indent=2))
REBUILD_DONE=True; print('\nCELL 2 OK')

---
## Cell 3 — Write everything and zip


In [ ]:
# ====================== CELL 3 — WRITE + ZIP ============================
if not globals().get('REBUILD_DONE'): raise SystemExit('>>> Run CELL 2 first.')

full.to_parquet(EXP/'rdd_frame.parquet')
frame[['project','rank','num_downloads','running','treated','age_days',
       'n_rel_pre','n_rel_post','days_to_next','abandoned']] \
     .to_parquet(EXP/'panel_frame.parquet')
rankdf.head(20000)[['project','num_downloads','rank']].to_parquet(EXP/'ranking_top20k.parquet')
panel.groupby(['month','treated'])['released'].agg(['mean','count']).reset_index() \
     .to_csv(EXP/'monthly_series.csv', index=False)
es.to_csv(EXP/'event_study_coefs.csv', index=False)
grids.to_csv(EXP/'robustness_grids.csv', index=False)
(EXP/'validation.json').write_text(json.dumps(validation, indent=2))
if 'removed' in raw2.columns:
    raw2[['project','removed']].to_parquet(EXP/'removal_flags.parquet')
(EXP/'late_entrants.txt').write_text('\n'.join(sorted(late)))

for f in ['results_rdd.json','results_panel.json','results_extended.json']:
    if (CACHE/f).exists(): (EXP/f).write_text((CACHE/f).read_text())
for pat in ['PRESPEC_*.json']:
    for f in CACHE.glob(pat): (EXP/f.name).write_text(f.read_text())
for f in ['fig_raw_trends.png','fig_event_study.png']:
    if (CACHE/f).exists():
        import shutil; shutil.copy2(CACHE/f, EXP/f)

zp = WORK/'paper_data.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(EXP.iterdir()): z.write(f, f.name)
print(f'WROTE {zp}  ({zp.stat().st_size/1e6:.1f} MB)')
for f in sorted(EXP.iterdir()): print('  ', f.name, f'{f.stat().st_size/1024:.0f} KB')